In [62]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\Minas"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\Minas\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "Minas do Sol - Eletromecanico - Taguemento.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "Minas do Sol - Eletromecanico - Taguemento.dxf")

temp_dwg_file = os.path.join(temp_folder, "Minas do Sol - Eletromecanico - Taguemento.dwg")

conn = load_to_duckdb(dxf_file, "PV Minas do Sol")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)


Project PV Minas do Sol already exists
Removed existing data for project: PV Minas do Sol
Successfully loaded project
Loaded 13472 entities for project: PV Minas do Sol
Don't forget to close connection.
Data Loaded
DuckDB Loaded


In [63]:
ELM = conn.sql("SELECT layer, name, X, Y, UFV, PST, INV, GRP, STR, rotation FROM cad_entities WHERE project_name='PV Minas do Sol'")

print(ELM)
print(conn.execute("DESCRIBE ELM").fetchdf())

result = conn.execute("""
    SELECT
        name,
        COUNT(*) as count
    FROM ELM
    GROUP BY name
    ORDER BY count DESC
""").fetchdf()
print(result)

┌─────────┬─────────────────────────┬──────────┬───────────┬─────────┬─────────┬─────────┬─────────┬─────────┬──────────┐
│  layer  │          name           │    X     │     Y     │   UFV   │   PST   │   INV   │   GRP   │   STR   │ rotation │
│ varchar │         varchar         │  float   │   float   │ varchar │ varchar │ varchar │ varchar │ varchar │  float   │
├─────────┼─────────────────────────┼──────────┼───────────┼─────────┼─────────┼─────────┼─────────┼─────────┼──────────┤
│ MDS_1.1 │ String_Tracker_1X174p_1 │ 511819.7 │ 8087402.5 │ 1       │ 1       │ 7       │ NULL    │         │      0.0 │
│ MDS_1.1 │ String_Tracker_1X174p_2 │ 511819.7 │ 8087368.5 │ 1       │ 1       │ 7       │ NULL    │         │      0.0 │
│ MDS_1.1 │ String_Tracker_1X174p_3 │ 511819.7 │ 8087334.5 │ 1       │ 1       │ 7       │ NULL    │         │      0.0 │
│ MDS_1.1 │ String_Tracker_1X174p_1 │ 511819.7 │ 8087230.5 │ 1       │ 1       │ 7       │ NULL    │         │    180.0 │
│ MDS_1.1 │ String_Track

In [64]:
strings = conn.sql("SELECT * FROM ELM WHERE name LIKE 'String_Tracker_%'")

strings_df = strings.fetchdf()

strings_df["GRP"] = strings_df["UFV"].astype(str) + "." + strings_df["PST"].astype(str) + "." + strings_df["INV"].astype(str).str.zfill(2)

strings_df['UFV'] = strings_df['UFV'].astype(int)
strings_df['PST'] = strings_df['PST'].astype(int)
strings_df['INV'] = strings_df['INV'].astype(int)
strings_df['Line'] = round(strings_df['X'],0)
print(strings_df.describe())
print(strings_df.info())
print(strings_df)

                   X             Y           UFV           PST           INV  \
count   12776.000000  1.277600e+04  12776.000000  12776.000000  12776.000000   
mean   513615.281250  8.088001e+06      4.506731      1.999843     14.039214   
std       927.287048  5.963423e+02      2.302750      0.817135      7.801798   
min    511649.687500  8.087006e+06      1.000000      1.000000      1.000000   
25%    513110.500000  8.087462e+06      2.000000      1.000000      7.000000   
50%    513566.093750  8.087992e+06      5.000000      2.000000     14.000000   
75%    514127.093750  8.088460e+06      7.000000      3.000000     21.000000   
max    519107.343750  8.089151e+06      8.000000      3.000000     27.000000   

           rotation          Line  
count  12776.000000   12776.00000  
mean      78.954292  513615.28125  
std       89.323105     927.30481  
min        0.000000  511650.00000  
25%        0.000000  513110.00000  
50%        0.000000  513566.00000  
75%      180.000000  514127

In [65]:
inverters = conn.sql("SELECT * FROM ELM WHERE name='MDS-LAYOUT$0$PRF_Inversor'")

inverter_df = inverters.fetchdf()

inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)
inverter_df.drop(columns=['name', 'rotation'], inplace=True)
inverter_df["GRP"] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str) + "." + inverter_df["INV"].astype(str).str.zfill(2)
inverter_df['inverter_line'] = round(inverter_df['inverter_x'],0)
inverter_df.sort_values(by=['inverter_y', 'inverter_x'], ascending=[False, True], inplace=True)
print(inverter_df.describe())
print(inverter_df.info())
print(inverter_df)

          inverter_x    inverter_y  inverter_line
count     648.000000  6.480000e+02     648.000000
mean   513613.718750  8.088022e+06  513613.718750
std       912.610168  5.789148e+02     912.626282
min    511656.500000  8.087182e+06  511656.000000
25%    513121.828125  8.087608e+06  513121.500000
50%    513568.750000  8.088079e+06  513569.000000
75%    514123.414062  8.088512e+06  514123.250000
max    515957.656250  8.088994e+06  515958.000000
<class 'pandas.DataFrame'>
Index: 648 entries, 614 to 173
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   layer          648 non-null    str    
 1   inverter_x     648 non-null    float32
 2   inverter_y     648 non-null    float32
 3   UFV            648 non-null    str    
 4   PST            648 non-null    str    
 5   INV            648 non-null    str    
 6   GRP            648 non-null    str    
 7   STR            648 non-null    str    
 8   inverter_line  

In [66]:
strings_df.sort_values(by=['Line', 'Y'], ascending=[True, False], inplace=True)


#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Different PSTs
condition2 = strings_df["GRP"] != strings_df["GRP"].shift()

#Break in the Structure
condition3 = abs(strings_df["Y"] - strings_df["Y"].shift()) > 45

break_condition_1 = condition1 | condition2 | condition3
break_condition_2 = condition1 | condition3

strings_df['SubGroup'] = break_condition_1.cumsum() # The SubGroup represents strings that are on the same Line, are from the same inverter and don't have a break between them. Used to count the number of strings that are going down the pile in the end.
strings_df['Group'] = break_condition_2.cumsum() # The Group represents strings that are on the same Line and don't have a break between them. Used to calculate where the start and end piles are.
print(strings_df)
strings_df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

         layer                     name             X          Y  UFV  PST  \
5769   MDS_1.1  String_Tracker_1X174p_1  511649.68750  8087177.5    1    1   
5770   MDS_1.1  String_Tracker_1X174p_2  511649.68750  8087143.5    1    1   
5771   MDS_1.1  String_Tracker_1X174p_3  511649.68750  8087110.0    1    1   
5774   MDS_1.1  String_Tracker_1X174p_3  511649.68750  8087074.0    1    1   
5773   MDS_1.1  String_Tracker_1X174p_2  511649.68750  8087040.0    1    1   
...        ...                      ...           ...        ...  ...  ...   
12772  MDS_3.3  String_Tracker_1X145p_1  519097.81250  8088930.0    3    3   
12768  MDS_3.3  String_Tracker_1X145p_2  519100.18750  8088930.0    3    3   
12769  MDS_3.3  String_Tracker_1X145p_3  519102.56250  8088930.0    3    3   
12770  MDS_3.3  String_Tracker_1X145p_4  519104.93750  8088930.0    3    3   
12771  MDS_3.3  String_Tracker_1X145p_5  519107.34375  8088930.0    3    3   

       INV     GRP STR  rotation      Line  SubGroup  Group  
5

In [67]:
distance1 = 15.8086 # Distance between the middle point of the string and the tracker's pile.

strings_df['N exit'] = strings_df['Y'] + distance1
strings_df['S exit'] = strings_df['Y'] - distance1

strings_df['SubGroup - Position Y - N'] = strings_df.groupby("SubGroup")["N exit"].transform('max')
strings_df['SubGroup - Position Y - S'] = strings_df.groupby("SubGroup")["S exit"].transform('min')

strings_df['Group - Position Y - N'] = strings_df.groupby("Group")["N exit"].transform('max')
strings_df['Group - Position Y - S'] = strings_df.groupby("Group")["S exit"].transform('min')

cables_df = strings_df.merge(inverter_df, on="GRP")

strings_df['No. of Strings'] = strings_df.groupby("SubGroup")['Line'].transform('count')

print(strings_df)

         layer                     name             X          Y  UFV  PST  \
5769   MDS_1.1  String_Tracker_1X174p_1  511649.68750  8087177.5    1    1   
5770   MDS_1.1  String_Tracker_1X174p_2  511649.68750  8087143.5    1    1   
5771   MDS_1.1  String_Tracker_1X174p_3  511649.68750  8087110.0    1    1   
5774   MDS_1.1  String_Tracker_1X174p_3  511649.68750  8087074.0    1    1   
5773   MDS_1.1  String_Tracker_1X174p_2  511649.68750  8087040.0    1    1   
...        ...                      ...           ...        ...  ...  ...   
12772  MDS_3.3  String_Tracker_1X145p_1  519097.81250  8088930.0    3    3   
12768  MDS_3.3  String_Tracker_1X145p_2  519100.18750  8088930.0    3    3   
12769  MDS_3.3  String_Tracker_1X145p_3  519102.56250  8088930.0    3    3   
12770  MDS_3.3  String_Tracker_1X145p_4  519104.93750  8088930.0    3    3   
12771  MDS_3.3  String_Tracker_1X145p_5  519107.34375  8088930.0    3    3   

       INV     GRP STR  rotation      Line  SubGroup  Group    

In [68]:
groups_df = strings_df.groupby("SubGroup")[['GRP','X', 'Y', 'SubGroup - Position Y - N', 'SubGroup - Position Y - S', 'Group - Position Y - N', 'Group - Position Y - S', 'No. of Strings', 'Line']].first()
groups_df.rename(columns={'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
print(f'Maximum number of strings in a group: {groups_df['No. of Strings'].max()}\n')
print(groups_df)

Maximum number of strings in a group: 7

             GRP             X          Y    Y-N-STR    Y-S-STR    Y-N-TRK  \
SubGroup                                                                     
1         1.1.13  511649.68750  8087177.5  8087193.5  8086990.0  8087193.5   
2         1.1.13  511656.50000  8087177.5  8087193.5  8086990.0  8087193.5   
3         1.1.13  511663.28125  8087177.5  8087193.5  8086990.0  8087193.5   
4         1.1.02  511670.09375  8087402.5  8087418.5  8087214.5  8087418.5   
5         1.1.13  511670.09375  8087177.5  8087193.5  8087127.5  8087193.5   
...          ...           ...        ...        ...        ...        ...   
2538      3.3.02  519097.81250  8088930.0  8088946.0  8088914.0  8088946.0   
2539      3.3.02  519100.18750  8088930.0  8088946.0  8088914.0  8088946.0   
2540      3.3.02  519102.56250  8088930.0  8088946.0  8088914.0  8088946.0   
2541      3.3.03  519104.93750  8088930.0  8088946.0  8088914.0  8088946.0   
2542      3.3.03  51910

In [69]:
duct_map = {
    1: '-1-',
    2: '-2-',
    3: '-3-',
    4: '-3-1-',
    5: '-3-2-',
    6: '-3-3-',
    7: '-3-3-1-',
    8: '-3-3-2-',
    9: '-3-3-3-',
    10: '-3-3-3-1-',
    11: '-3-3-3-2-',
    12: '-3-3-3-3-',
    13: '-3-3-3-3-1-',
    14: '-3-3-3-3-2-',
    15: '-3-3-3-3-3-',
    16: '-3-3-3-3-3-1-',
    17: '-3-3-3-3-3-2-',
    18: '-3-3-3-3-3-3-',
    19: '-3-3-3-3-3-3-1-',
    20: '-3-3-3-3-3-3-2-',
}

groups_df.loc[:,'Duct Definition'] = groups_df['No. of Strings'].map(duct_map)
groups_df['Number of Ducts'] = groups_df['Duct Definition'].astype(str).str.count('-')-1
groups_df = groups_df.merge(inverter_df, on='GRP')
groups_df['N or S or M'] = np.where(
    abs(groups_df['Y-N-TRK'] - groups_df['inverter_y']) > abs(groups_df['Y-S-TRK'] - groups_df['inverter_y']),
    "S",
    np.where(
        (groups_df['inverter_y'] < groups_df['Y-N-TRK']) & (groups_df['inverter_y'] > groups_df['Y-S-TRK']),
        "M",
        "N"
    )
)

# THIS FINDS THE CLOSEST Y TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (Y-N-TRK, Y-S-TRK OR Y).
groups_df['closest_y'] = np.where(
    abs(groups_df['Y-N-TRK'] - groups_df['inverter_y']) < abs(groups_df['Y'] - groups_df['inverter_y']),
    groups_df['Y-N-TRK'],
    np.where(
        abs(groups_df['Y'] - groups_df['inverter_y']) < abs(groups_df['Y-S-TRK'] - groups_df['inverter_y']),
        groups_df['Y'],
        groups_df['Y-S-TRK']
    )
)

groups_df.sort_values(['GRP', 'Line'], inplace = True)

print(groups_df.head(20))


       GRP             X          Y    Y-N-STR    Y-S-STR    Y-N-TRK  \
43  1.1.01  511778.90625  8087264.5  8087280.5  8087214.5  8087418.5   
45  1.1.01  511785.68750  8087402.5  8087418.5  8087248.5  8087418.5   
47  1.1.01  511792.50000  8087402.5  8087418.5  8087248.5  8087418.5   
49  1.1.01  511799.28125  8087402.5  8087418.5  8087248.5  8087418.5   
51  1.1.01  511806.09375  8087402.5  8087418.5  8087317.5  8087418.5   
3   1.1.02  511670.09375  8087402.5  8087418.5  8087214.5  8087418.5   
6   1.1.02  511676.90625  8087402.5  8087418.5  8087214.5  8087418.5   
8   1.1.02  511683.68750  8087402.5  8087418.5  8087214.5  8087418.5   
10  1.1.02  511690.50000  8087402.5  8087418.5  8087352.5  8087418.5   
11  1.1.03  511690.50000  8087334.5  8087350.5  8087214.5  8087418.5   
14  1.1.03  511697.28125  8087402.5  8087418.5  8087214.5  8087418.5   
16  1.1.03  511704.09375  8087402.5  8087418.5  8087214.5  8087418.5   
18  1.1.03  511710.90625  8087402.5  8087418.5  8087282.5  80874

Breaking the trenches into segments to indicate how many strings there are in each one.
=

In [70]:
"""
Adds cumulative sum column where values sum towards the inverter_line, calculated separately for each distinct group/segment.
"""
groups_df['STR_CUM_TO_INV'] = 0

# Group by GRP
# Each GRP represents a distinct physical segment with its own inverter
for grp_name in groups_df['GRP'].unique():
    # Get rows belonging to this GRP
    grp_mask = groups_df['GRP'] == grp_name
    grp_indices = groups_df[grp_mask].index

    if len(grp_indices) == 0:
        continue

    # Get the inverter_line for this GRP (should be consistent within group)
    # Using the first occurrence's inverter_line as the target for this group
    inv_line = groups_df.loc[grp_indices[0], 'inverter_line']

    # Separate rows by position relative to inverter_line
    before_inv = []
    at_inv = []
    after_inv = []

    for idx in grp_indices:
        x_val = groups_df.loc[idx, 'Line']
        if x_val < inv_line:
            before_inv.append(idx)
        elif x_val > inv_line:
            after_inv.append(idx)
        else:  # x_val == inv_line
            at_inv.append(idx)

    # Sort before_inv by X ascending (farthest to closest to inverter)
    before_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'])

    # Sort after_inv by X descending (farthest to closest to inverter)
    after_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'], reverse=True)

    # Calculate cumulative sums for before inverter (left to right)
    cumsum = 0
    for idx in before_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum

    # At inverter line: just the value itself
    for idx in at_inv:
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = groups_df.loc[idx, 'No. of Strings']

    # Calculate cumulative sums for after inverter (right to left)
    cumsum = 0
    for idx in after_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum


# Verify the results
print(groups_df[['GRP', 'X', 'No. of Strings', 'inverter_line', 'STR_CUM_TO_INV']].to_string())
max_number_of_strings = groups_df['STR_CUM_TO_INV'].max()
print(groups_df['STR_CUM_TO_INV'].value_counts())
groups_df.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")

         GRP             X  No. of Strings  inverter_line  STR_CUM_TO_INV
43    1.1.01  511778.90625               2       511792.0               2
45    1.1.01  511785.68750               5       511792.0               7
47    1.1.01  511792.50000               5       511792.0               5
49    1.1.01  511799.28125               5       511792.0               8
51    1.1.01  511806.09375               3       511792.0               3
3     1.1.02  511670.09375               6       511677.0               6
6     1.1.02  511676.90625               6       511677.0               6
8     1.1.02  511683.68750               6       511677.0               8
10    1.1.02  511690.50000               2       511677.0               2
11    1.1.03  511690.50000               4       511704.0               4
14    1.1.03  511697.28125               6       511704.0              10
16    1.1.03  511704.09375               6       511704.0               6
18    1.1.03  511710.90625            

In [71]:
distance2 = 0.5618 # Distance between the first/last pile and the first/last module

cables_df.drop(columns=['UFV_x', 'UFV_y', 'PST_x', 'PST_y', 'INV_x', 'INV_y', 'SubGroup', 'Group'], inplace=True)
cables_df.rename(columns={'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
cables_df['N or S or M'] = np.where(
    abs(cables_df['N exit'] - cables_df['inverter_y']) > abs(cables_df['S exit'] - cables_df['inverter_y']),
    "S",
    np.where(
        (cables_df['inverter_y'] < cables_df['N exit']) & (cables_df['inverter_y'] > cables_df['S exit']),
        "M",
        "N"
    )
)

# THIS FINDS THE CLOSEST Y TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (Y-N-TRK, Y-S-TRK OR Y).
cables_df['closest_y'] = np.where(
    abs(cables_df['Y-N-TRK'] - cables_df['inverter_y']) < abs(cables_df['Y'] - cables_df['inverter_y']),
    cables_df['Y-N-TRK'],
    np.where(
        abs(cables_df['Y'] - cables_df['inverter_y']) < abs(cables_df['Y-S-TRK'] - cables_df['inverter_y']),
        cables_df['Y'],
        cables_df['Y-S-TRK']
    )
)


cables_df['Y-N-STR'] = cables_df['Y-N-STR'] + distance2
cables_df['Y-S-STR'] = cables_df['Y-S-STR'] - distance2

cables_df.sort_values(['GRP', 'Line'], inplace = True)
print(cables_df.head())

     layer_x                     name             X          Y     GRP STR_x  \
214  MDS_1.1  String_Tracker_1X174p_2  511778.90625  8087264.5  1.1.01         
215  MDS_1.1  String_Tracker_1X174p_1  511778.90625  8087230.5  1.1.01         
222  MDS_1.1  String_Tracker_1X145p_1  511785.68750  8087402.5  1.1.01         
223  MDS_1.1  String_Tracker_1X145p_2  511785.68750  8087368.5  1.1.01         
224  MDS_1.1  String_Tracker_1X145p_3  511785.68750  8087333.5  1.1.01         

     rotation      Line     N exit     S exit  ...    Y-S-STR    Y-N-TRK  \
214     180.0  511779.0  8087280.5  8087248.5  ...  8087214.0  8087418.5   
215     180.0  511779.0  8087246.5  8087214.5  ...  8087214.0  8087418.5   
222       0.0  511786.0  8087418.5  8087386.5  ...  8087248.0  8087418.5   
223       0.0  511786.0  8087384.5  8087352.5  ...  8087248.0  8087418.5   
224       0.0  511786.0  8087349.5  8087317.5  ...  8087248.0  8087418.5   

       Y-S-TRK  layer_y inverter_x  inverter_y  STR_y inverter

In [72]:
script_df = groups_df[['GRP','No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_y', 'X', 'inverter_x', 'inverter_y','Line', 'inverter_line', 'STR_CUM_TO_INV', 'Y-N-STR', 'Y-S-STR', 'N or S or M']].copy()
script_df.rename(columns={'closest_y': 'Y'}, inplace=True)

distance3 = 2.6208 # Vertical distance between the tracker's pile and the inverter.
distance4 = 0.675 # Distance between the middle point of the inverter and its edge on the X axis.
distance5 = 0.1825 # Distance between the middle point of the inverter and its edge on the Y axis.
distance6 = 1.5 # Distance to make an intermediary point for stretching purposes.
distance7 = 1.25 # Distance to make an intermediary point for stretching purposes.
distance8 = 1 # Distance to make an intermediary point for stretching purposes.
distance9 = 0.75 # Distance to make an intermediary point for stretching purposes.

def intermediary_coordinates(df):
    df['intermediary_x_1'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['X'] + distance3,
            df['X'] - distance3
        )
    )

    df['intermediary_x_2'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance6,
            df['inverter_x'] + distance6
        )
    )

    df['intermediary_x_3'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance7,
            df['inverter_x'] + distance7
        )
    )

    df['intermediary_x_4'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance8,
            df['inverter_x'] + distance8
        )
    )

    df['intermediary_x_5'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance9,
            df['inverter_x'] + distance9
        )
    )

    df['inverter_x'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance4,
            df['inverter_x'] + distance4
        )
    )

    df['inverter_y'] = np.where(
        df['inverter_line'] != df['Line'],
        df['inverter_y'],
        np.where(
            df['N or S or M'] == 'S',
            df['inverter_y'] + distance5,
            df['inverter_y'] - distance5
        )
    )

'''

CABOS

'''

intermediary_coordinates(cables_df)

def calculate_cable_length(row, color):
    try:
        x = row['X']
        y_n_str = row['N exit']
        y_s_str = row['S exit']
        closest_y = row['closest_y']
        int_x_1 = row['intermediary_x_1']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']

        if color == 'red':
            seg1 = math.sqrt((x - x)**2 + (y_n_str - closest_y)**2)
        else:
            seg1 = math.sqrt((x - x)**2 + (y_s_str - closest_y)**2)
        seg2 = math.sqrt((x - int_x_1)**2 + (closest_y - inv_y)**2)
        seg3 = math.sqrt((int_x_1 - inv_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2 + seg3
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise

cables_df['Red Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'red'), axis=1)
cables_df['Black Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'black'), axis=1)

total_red_length = cables_df['Red Cable Length'].sum()
total_black_length = cables_df['Black Cable Length'].sum()

total_cable_length = total_red_length + total_black_length

cables_df['Script Red Cables'] = cables_df.apply(
    lambda row: f"_.pline {row['X']},{row['N exit']} {row['X']},{row['closest_y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
cables_df['Script Black Cables'] = cables_df.apply(
    lambda row: f"_.pline {row['X']},{row['S exit']} {row['X']},{row['closest_y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\minas\Script Cabos Tracker.scr", 'w') as f:
    f.write('PLINEWID 0\n')
    f.write("LAYER S EMF_cabo_solar_positivo\n\n")
    f.write('\n'.join(cables_df["Script Red Cables"].astype(str)) + '\n')
    f.write("LAYER S EMF_cabo_solar_negativo\n\n")
    f.write('\n'.join(cables_df["Script Black Cables"].astype(str)) + '\n')

'''

VALAS

'''

intermediary_coordinates(script_df)

script_df['Script Valas'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
print(script_df['Script Valas'])

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\minas\Script Valas Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    f.write("LAYER S EMF_bt_vala\n\n")
    f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')


'''

DUTOS

'''

def calculate_duct_length(row, no_of_strings):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x_1']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']
        number_of_ducts = row['Duct Definition'].count(no_of_strings)

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - inv_x)**2 + (inv_y - inv_y)**2)

        return (seg1 + seg2) * number_of_ducts
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise


script_df['Dutos 1 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '1'), axis=1)
script_df['Dutos 2 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '2'), axis=1)
script_df['Dutos 3 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '3'), axis=1)
script_df['Dutos 4 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '4'), axis=1)

total_1str_duct_length = script_df['Dutos 1 Length'].sum()
total_2str_duct_length = script_df['Dutos 2 Length'].sum()
total_3str_duct_length = script_df['Dutos 3 Length'].sum()
total_4str_duct_length = script_df['Dutos 4 Length'].sum()

total_duct_length = total_1str_duct_length + total_2str_duct_length + total_3str_duct_length + total_4str_duct_length

script_df['Script Dutos 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('1'),
    axis=1
)
script_df['Script Dutos 2'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('2'),
    axis=1
)
script_df['Script Dutos 3'] = script_df.apply(
    lambda row: '\n'.join([
        f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} "
        for _ in range(str(row['Duct Definition']).count('3'))
    ]),
    axis=1
)
script_df['Script Dutos 4'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_4']},{row['inverter_y']} {row['intermediary_x_5']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('4'),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\minas\Script Dutos Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.03\n')
    # Layer 1
    layer1_commands = script_df["Script Dutos 1"].astype(str)
    layer1_commands = layer1_commands[layer1_commands != '']  # Remove empty strings
    if len(layer1_commands) > 0:  # Only write layer if there are commands
        f.write("LAYER S EMF_bt_eletroduto_1S\n\n")
        f.write('\n'.join(layer1_commands) + '\n')

    # Layer 2
    layer2_commands = script_df["Script Dutos 2"].astype(str)
    layer2_commands = layer2_commands[layer2_commands != '']
    if len(layer2_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_2S\n\n")
        f.write('\n'.join(layer2_commands) + '\n')

    # Layer 3
    layer3_commands = script_df["Script Dutos 3"].astype(str)
    layer3_commands = layer3_commands[layer3_commands != '']
    if len(layer3_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_3S\n\n")
        f.write('\n'.join(layer3_commands) + '\n')

    # Layer 4
    layer4_commands = script_df["Script Dutos 4"].astype(str)
    layer4_commands = layer4_commands[layer4_commands != '']
    if len(layer4_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_4S\n\n")
        f.write('\n'.join(layer4_commands) + '\n')

43      _.pline 511778.90625,8087264.5 511781.53125,80...
45      _.pline 511785.6875,8087248.5 511788.3125,8087...
47      _.pline 511792.5,8087248.5 511792.5,8087246.0 ...
49      _.pline 511799.28125,8087248.5 511796.65625,80...
51      _.pline 511806.09375,8087248.5 511803.46875,80...
                              ...                        
2439    _.pline 515692.46875,8088001.5 515689.84375,80...
2417    _.pline 515631.25,8087995.0 515633.875,8087998...
2419    _.pline 515638.0625,8087998.0 515640.6875,8087...
2421    _.pline 515644.875,8088001.0 515644.875,808799...
2423    _.pline 515651.65625,8088004.0 515649.03125,80...
Name: Script Valas, Length: 2542, dtype: str


In [73]:
'''

LAYERED VALAS

'''

pitch = 6.8

script_df['intermediary_x_6'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df['intermediary_x_1'] + pitch,
        script_df['intermediary_x_1'] - pitch
    )
)

script_df['final_x'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df.apply(lambda row: min(row['intermediary_x_6'], row['inverter_x']), axis=1),
        script_df.apply(lambda row: max(row['intermediary_x_6'], row['inverter_x']), axis=1)
    )
)

def calculate_trench_length(row):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x_1']
        inv_y = row['inverter_y']
        final_x = row['final_x']

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - final_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise


script_df['Valas Length'] = script_df.apply(calculate_trench_length, axis=1)

total_trench_length = script_df['Valas Length'].sum()

script_df['Script Valas Layered 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} ",
    axis=1
)

script_df['Script Valas Layered 2'] = script_df.apply(
    lambda row: f"_.pline {row['intermediary_x_1']},{row['inverter_y']} {row['final_x']},{row['inverter_y']} ",
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\minas\Script Valas Layered Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    for i in range(1, max_number_of_strings+1):
        mask1 = script_df['No. of Strings'] == i
        mask2 = script_df['STR_CUM_TO_INV'] == i

        if mask1.any() or mask2.any():
            f.write(f"LAYER S EMF_vala_CC_{i}C\n\n")

            if mask1.any():
                f.write('\n'.join(script_df[mask1]["Script Valas Layered 1"].astype(str)) + '\n')

            if mask2.any():
                f.write('\n'.join(script_df[mask2]["Script Valas Layered 2"].astype(str)) + '\n')


lengths_df = pd.DataFrame.from_dict(data={
    'DC Red Cable Length': [total_red_length],
    'DC Black Cable Length': [total_black_length],
    'DC Total Cable Length': [total_cable_length],
    '1str Duct Length': [total_1str_duct_length],
    '2str Duct Length': [total_2str_duct_length],
    '3str Duct Length': [total_3str_duct_length],
    '4str Duct Length': [total_4str_duct_length],
    'Total Duct Length': [total_duct_length],
    'Total Trench Length': [total_trench_length]
}, orient="index", columns=['Length (m)'])
lengths_df.to_excel(r"C:\Users\Usuario\Desktop\Lengths.xlsx")
script_df.to_excel(r"C:\Users\Usuario\Desktop\SCRIPTS.xlsx")
cables_df.to_excel(r"C:\Users\Usuario\Desktop\cables.xlsx")

conn.close()